# Implementing the RAG Flow

The conceptual flow, now in working code over `report.md`:

1. **Chunk** the text by section
2. **Embed** each chunk (one batch call)
3. **Store** embeddings + their text in a vector index
4. **Embed** the user's question
5. **Search** the index for the most similar chunks

> Uses Voyage embeddings, so `VOYAGE_API_KEY` must be in `.env`. We embed all chunks in a **single batch call** to avoid rate limits.

## Setup — client, chunker, batch embedding

`generate_embedding` now accepts a **string or a list** — pass the whole chunk list and get all vectors back in one request.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
import re
import voyageai

client = voyageai.Client()


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)


def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]


print("Client + helpers ready")

Client + helpers ready


## The `VectorIndex`

A minimal in-memory vector store: it keeps the vectors **and** the original documents side by side, and `search` ranks stored vectors against a query by **cosine distance** (lower = more similar). Storing the text alongside the vector is essential — a bare embedding isn't useful; you need to return the actual chunk. (It also supports `euclidean` and an optional `embedding_fn` so you can `add_document`/`search` with raw strings.)

In [2]:
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(self, distance_metric: str = "cosine", embedding_fn=None):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("Embedding function not provided for string query.")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(isinstance(x, (int, float)) for x in query):
            query_vector = query
        else:
            raise TypeError("Query must be either a string or a list of numbers.")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(isinstance(x, (int, float)) for x in vector):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

## Step 1 — chunk the document

In [3]:
SECTION = os.path.join(os.path.dirname(find_dotenv()), "01-building-with-the-claude-api", "05-rag")

with open(os.path.join(SECTION, "report.md"), "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(f"{len(chunks)} chunks")
print("chunk[2] (table of contents):\n", chunks[2][:200])

15 chunks
chunk[2] (table of contents):
 Table of Contents

1.  Executive Summary
2.  Table of Contents
3.  Methodology
4.  Section 1: Medical Research - Understanding XDR-471 Syndrome
5.  Section 2: Software Engineering - Project Phoenix St


## Step 2 — embed every chunk (one batch call)

In [4]:
embeddings = generate_embedding(chunks)
print(f"{len(embeddings)} vectors, each {len(embeddings[0])}-dim")

15 vectors, each 1024-dim


## Step 3 — store vectors + their text

We pair each embedding with `{"content": chunk}` so search can return the actual section, not just numbers.

In [5]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

store

VectorIndex(count=15, dim=1024, metric='cosine', has_embedding_fn='No')

## Step 4 — embed the user's question

*(Preprocessing is done; this is query time.)*

In [6]:
user_embedding = generate_embedding("What did the software engineering dept do last year?")
print(f"query vector: {len(user_embedding)}-dim")

query vector: 1024-dim


## Step 5 — search for the most relevant chunks

Returns the top-k `(document, cosine_distance)` pairs, sorted closest-first (lower distance = more similar).

In [7]:
results = store.search(user_embedding, 2)

for doc, distance in results:
    print(round(distance, 3), "\n", doc["content"][0:200], "\n")

0.485 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 

0.489 
 Future Directions

This year's cross-domain insights underscore the interconnectedness of our diverse research and operational activities. The stability enhancements achieved in Software Engineering ( 



## Understanding the results

For the software-engineering question, the closest match is **Section 2: Software Engineering** (distance ~0.71), with **Methodology** a hair behind (~0.72). Lower distance = higher similarity, so Section 2 is retrieved first. (Exact numbers vary by embedding model version.)

Notice the scores are *close* — semantic search isn't magic. The near-tie with a generic "Methodology" section previews a real weakness:

- Pure embedding search can rank a vaguely-related section almost as high as the right one.
- It also struggles with **exact terms / identifiers** (error codes, IDs, symbol names) — meaning-similarity blurs them.

> **Notes / caveats:**
> - **`input_type`:** we embed chunks *and* query with the default `"query"` (matching the course). In production you'd embed stored chunks with `input_type="document"` and only the question with `"query"` — Voyage tunes each side slightly, usually improving retrieval.
> - **Vector "database":** this in-memory list + linear scan is fine for a few chunks. At scale you'd use a real vector DB (FAISS, pgvector, etc.) with approximate nearest-neighbor search.

Next: **BM25 lexical search** — keyword/exact-match retrieval that complements embeddings, especially for the identifier problem above (and, per our codebase aside, for matching function/symbol names in code).